#### Environment Setup & Raw Data Ingestion

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Layer, Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam

print("Loading datasets...")
# Using your exact local paths
all_data_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx"
breakdown_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx"

df_all = pd.read_excel(all_data_file)
df_breakdown = pd.read_excel(breakdown_file)

# Handle column spelling differences
col_healthy = 'Breakdown' if 'Breakdown' in df_all.columns else 'Brekadown'
col_broken = 'Breakdown' if 'Breakdown' in df_breakdown.columns else 'Brekadown'

# Isolate the healthy data 
df_healthy = df_all[df_all[col_healthy].isna()].copy()
print(f"Loaded {len(df_healthy)} healthy records and {len(df_breakdown)} breakdown records.")

#### Feature Engineering

In [ ]:
print("Combining Vibration and Electrical Data...")

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# 🚩 UPDATE: machineRPM has been successfully removed from this list
extra_features = [ 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]

# Process Healthy Data
h_vibs = pd.DataFrame(df_healthy['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
h_extras = df_healthy[extra_features].reset_index(drop=True)
df_h_ml = pd.concat([h_extras, h_vibs], axis=1)
df_h_ml['Label'] = 'Healthy Baseline'

# Process Breakdown Data
b_vibs = pd.DataFrame(df_breakdown['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
b_extras = df_breakdown[extra_features].reset_index(drop=True)
df_b_ml = pd.concat([b_extras, b_vibs], axis=1)
df_b_ml['Label'] = df_breakdown[col_broken].values

# Combine into one Master Dataset
df_ml_master = pd.concat([df_h_ml, df_b_ml]).fillna(0)
print(f"Master Dataset created! The AI will look at {df_ml_master.shape[1] - 1} different variables per machine.")

In [ ]:
df_ml_master

#### Label Encoding & Feature Standardization

In [ ]:
print("Formatting Data for Deep Learning...")

X_raw = df_ml_master.drop(columns=['Label']).values
y_text = df_ml_master['Label'].values

# Encode text labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_text)
y_categorical = to_categorical(y_encoded)

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

with open('sewing_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('sewing_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
    
print("Translators saved successfully.")

#### Time-Series Transformation

In [ ]:
print("Creating Sliding Time Windows...")

def create_sequences(data_X, data_Y, time_steps=5):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - time_steps):
        X_seq.append(data_X[i : (i + time_steps)])
        y_seq.append(data_Y[i + time_steps]) 
    return np.array(X_seq), np.array(y_seq)

TIME_STEPS = 5 
X_lstm, y_lstm = create_sequences(X_scaled, y_categorical, time_steps=TIME_STEPS)

# 🚩 CRITICAL FIX: shuffle=False prevents Data Leakage in Time Series
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42, shuffle=False)

print(f"LSTM Input Shape: {X_train.shape}")

#### Deep Learning Architecture Definition(Attention-Augmented LSTM)

In [ ]:
print("🧠 Building the Novelty: Attention-Augmented LSTM...")


# 1. DEFINE THE CUSTOM ATTENTION LAYER
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), 
                                 initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), 
                                 initializer='zeros', trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1) # The Attention Weights
        output = x * a
        return K.sum(output, axis=1)

# 2. BUILD THE ARCHITECTURE (Functional API)
inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

lstm_out = LSTM(64, return_sequences=True)(inputs)
lstm_out = Dropout(0.2)(lstm_out)

# The Attention Spotlight
attention_out = AttentionLayer()(lstm_out)

dense_out = Dense(32, activation='relu')(attention_out)
dense_out = Dropout(0.2)(dense_out)

num_classes = y_categorical.shape[1]
outputs = Dense(num_classes, activation='softmax')(dense_out)

# 3. COMPILE
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

#### Model Training & Evaluation

In [ ]:
print("Commencing Model Training...")

history = model.fit(
    X_train, y_train,
    epochs=50,           
    batch_size=16,       
    validation_data=(X_test, y_test), 
    verbose=1
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"FINAL REAL-WORLD ACCURACY: {accuracy * 100:.2f}%")
print("="*40)

model.save("sewing_machine_predictive_attention_lstm.h5")
print("✅ SUCCESS! Saved the .h5 model to your computer.")

#### The Ultimate Objective (Live Inference & Solution Engine)

In [ ]:
print("\n--- LIVE INFERENCE & RULE-BASED SOLUTION ENGINE ---")

# 1. The Standard Operating Procedures Dictionary
maintenance_solutions = {
    "Needle Breakage": "SOLUTION: Immediately replace the needle and check the needle guard alignment.",
    "Thread Jam": "SOLUTION: Clear the bobbin case, check thread tension, and clean the feed dogs.",
    "Blade Blunt": "SOLUTION: Schedule a replacement of the upper and lower cutting blades within 1 hour.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial by 2 turns.",
    "Healthy Baseline": "STATUS: Machine operating normally. No action required."
    # Add the rest of your 15 specific breakdowns here!
}

# 2. Simulate pulling the most recent 5-second window of live factory data
sample_window = X_test[0:1] 

# 3. AI Prediction
prediction_probs = model.predict(sample_window, verbose=0)
predicted_label_index = np.argmax(prediction_probs)
predicted_breakdown = encoder.inverse_transform([predicted_label_index])[0]

# 4. Actionable Output
print(f"⚠️ PREDICTED STATE: {predicted_breakdown}")

if predicted_breakdown in maintenance_solutions:
    print(f"🔧 {maintenance_solutions[predicted_breakdown]}")
else:
    print("🔧 SOLUTION: Standard mechanical inspection required.")